## 0. Setup — Download Dataset dari Roboflow

Notebook ini akan download dataset langsung dari Roboflow ke Colab.

Link: `https://app.roboflow.com/ds/qFlbnnIoI8?key=jjFHrlZqjO`

Kalau download gagal, kamu bisa Upload zip manual atau pakai dataset dari Google Drive.

In [ ]:
# ============================================
# Langkah 1: Download dataset dari Roboflow
# ============================================
from google.colab import drive
from google.colab import files
from pathlib import Path
import zipfile
import os

# Mount Drive (opsional, untuk menyimpan model hasil training)
drive.mount('/content/drive')

# Konfigurasi
ROBOFLOW_URL = "https://app.roboflow.com/ds/qFlbnnIoI8?key=jjFHrlZqjO"
DRIVE_PATH = Path('/content/drive/MyDrive/ultra-milk-yolo-ready')
CONTENT_PATH = Path('/content/ultra-milk-yolo-ready')
ZIP_PATH = Path('/content/roboflow.zip')

def ensure_dataset(base: Path) -> Path:
    """Verifikasi data.yaml ada di base path."""
    if base.exists() and (base / 'data.yaml').exists():
        return base
    return None

def rename_classes(data_yaml_path: Path) -> None:
    """Rename kelas Roboflow menjadi QC SortVision."""
    text = data_yaml_path.read_text(encoding='utf-8')
    old = "names: ['um_normal', 'um_rusak']"
    new = "names: ['passed', 'damaged']"
    if old in text:
        text = text.replace(old, new)
        data_yaml_path.write_text(text, encoding='utf-8')
        print('Class names renamed: um_normal -> passed, um_rusak -> damaged')
    else:
        print('Class names already renamed or different format')

# Prioritas 1: Download dari Roboflow
print(f"Mendownload dataset dari Roboflow...")
os.system(f'curl -L "{ROBOFLOW_URL}" -o {ZIP_PATH}')

if ZIP_PATH.exists() and ZIP_PATH.stat().st_size > 1000:
    print(f"Mengekstrak {ZIP_PATH}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall('/content/')
    ZIP_PATH.unlink()
    
    # Cari folder yang berisi data.yaml
    base = ensure_dataset(CONTENT_PATH)
    if base is None:
        # Roboflow bisa ekstrak ke subfolder atau langsung ke /content
        search_path = CONTENT_PATH if CONTENT_PATH.exists() else Path('/content')
        for subdir in search_path.iterdir():
            if subdir.is_dir() and ensure_dataset(subdir):
                base = subdir
                break
        if base is None and ensure_dataset(Path('/content')):
            base = Path('/content')
    
    if base is not None:
        rename_classes(base / 'data.yaml')
        print(f'Dataset siap di: {base}')
    else:
        raise FileNotFoundError('data.yaml tidak ditemukan setelah ekstrak Roboflow.')
else:
    raise FileNotFoundError('Gagal download dari Roboflow.')

# Set global paths
DATA_BASE = str(base)
DATA_YAML = str(base / 'data.yaml')

print('\nKonfigurasi dataset:')
print(f'  Base   : {DATA_BASE}')
print(f'  YAML   : {DATA_YAML}')


## 1. Install Ultralytics + Cek GPU

In [ ]:
%pip install -q ultralytics

import torch
print('PyTorch version    :', torch.__version__)
print('CUDA available     :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU                :', torch.cuda.get_device_name(0))


## 2. Train Model YOLOv8

In [ ]:
from ultralytics import YOLO

use_cuda = torch.cuda.is_available()

print('Data yaml  :', DATA_YAML)
print('Epochs     :', 10)
print('Device     :', 'cuda' if use_cuda else 'cpu')

model = YOLO('yolov8n.pt')
model.train(
    data=DATA_YAML,
    epochs=10,
    imgsz=640,
    batch=16,
    device='cuda' if use_cuda else 'cpu',
    workers=2,
    cache=False,
    project='/content/runs/detect',
    name='train',
    exist_ok=True,
)


## 3. Download best.pt

In [ ]:
from google.colab import files
from pathlib import Path

best = Path('/content/runs/detect/train/weights/best.pt')
if not best.exists():
    raise FileNotFoundError(f'best.pt tidak ditemukan di {best}. Pastikan training selesai.')

files.download(str(best))
print('Downloaded:', best)
